# Gradient Boosting Final: Referral Success Prediction

**Project deliverable ? final ML engine logic**  
Approach: `HistGradientBoostingClassifier` (gradient boosting)  
Dataset: Medical Appointment No-Shows, Brazil (2016)

## Objective

Predict if a booked appointment will be missed. The target is `no_show` (`1 = missed`, `0 = attended`); the application reports success as `1 ? P(no_show)`. This notebook records the full workflow: clean and explore the data, build referral-time features, train and compare against a baseline, select a threshold on validation data, evaluate once on later data, then save the API-compatible model bundle.

**Gradient boosting, simply:** it adds small decision trees one after another. Each new tree focuses on errors left by earlier trees; the combined trees produce the final risk estimate. Learning-rate and tree-size limits help control overfitting.

The source field `SMS_received` is valid only if a reminder is already planned at scoring time. In the application, this corresponds to `reminder_planned`.


## 1. Setup and load the sample data

Install `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn` (1.2 or newer), and `joblib` in the notebook environment. The full Kaggle file is not included in this two-notebook submission. Download `KaggleV2-May-2016.csv` and place it at the project root or in `data/`; the small `df-sample.csv` is for exploration only and is too small for chronological model evaluation.


In [ ]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
TARGET_RECALL = 0.70
MODEL_PATH = Path("referral_success_model.joblib")  # keep in sync with main.py
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

# Accept the full dataset at the project root or in data/.
DATA_CANDIDATES = [
    Path("data/KaggleV2-May-2016.csv"),
    Path("KaggleV2-May-2016.csv"),
    Path("../data/KaggleV2-May-2016.csv"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Download the full KaggleV2-May-2016.csv dataset and place it in the project root or data/ folder."
    )

raw = pd.read_csv(DATA_PATH)
print(f"Loaded {len(raw):,} rows and {raw.shape[1]} columns from {DATA_PATH}")
raw.head()


## 2. Clean records and define the target

In [ ]:
df = raw.rename(columns={
    "Hipertension": "Hypertension",
    "Handcap": "Handicap",
    "No-show": "NoShow",
    "SMS_received": "SMSReceived",
}).copy()

required = {
    "PatientId", "AppointmentID", "Gender", "ScheduledDay", "AppointmentDay", "Age",
    "Neighbourhood", "Scholarship", "Hypertension", "Diabetes", "Alcoholism",
    "Handicap", "SMSReceived", "NoShow",
}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Dataset is missing expected columns: {sorted(missing)}")

df["ScheduledDay"] = pd.to_datetime(df["ScheduledDay"], utc=True, errors="coerce")
df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"], utc=True, errors="coerce")
df["no_show"] = df["NoShow"].map({"Yes": 1, "No": 0})
df["sched_date"] = df["ScheduledDay"].dt.normalize()
df["appt_date"] = df["AppointmentDay"].dt.normalize()
df["lead_days"] = (df["appt_date"] - df["sched_date"]).dt.days

before = len(df)
df = df.dropna(subset=["ScheduledDay", "AppointmentDay", "no_show", "Age", "PatientId"])
df = df[df["Age"].between(0, 110) & (df["lead_days"] >= 0)].copy()
df.reset_index(drop=True, inplace=True)
df["is_female"] = (df["Gender"] == "F").astype("int8")
df["Handicap"] = df["Handicap"].clip(upper=1)
df["same_day"] = (df["lead_days"] == 0).astype("int8")
df["sched_hour"] = df["ScheduledDay"].dt.hour
df["sched_weekday"] = df["ScheduledDay"].dt.day_name()
df["appt_weekday"] = df["AppointmentDay"].dt.day_name()
print(f"Removed {before - len(df):,} invalid rows; {len(df):,} usable rows remain.")
print(f"No-show rate: {df['no_show'].mean():.1%}; attendance/success rate: {1 - df['no_show'].mean():.1%}")


## 3. Build referral-time patient history (without outcome leakage)

For each referral, history includes only appointments whose appointment date is strictly before the booking date. Their outcomes would therefore have happened before the current referral was booked. Appointments on the same date are excluded from one another.

In [ ]:
daily = (
    df.groupby(["PatientId", "appt_date"], as_index=False)["no_show"]
      .agg(n="size", ns="sum")
      .sort_values(["PatientId", "appt_date"])
)
daily["cum_appts"] = daily.groupby("PatientId")["n"].cumsum()
daily["cum_noshows"] = daily.groupby("PatientId")["ns"].cumsum()

left = df[["PatientId", "sched_date"]].copy()
left["_row"] = np.arange(len(left))
left = left.sort_values(["sched_date", "PatientId"])
right = daily[["PatientId", "appt_date", "cum_appts", "cum_noshows"]].sort_values(["appt_date", "PatientId"])
history = pd.merge_asof(
    left, right,
    left_on="sched_date", right_on="appt_date", by="PatientId",
    direction="backward", allow_exact_matches=False,
).sort_values("_row")

df["prior_appts"] = history["cum_appts"].fillna(0).astype("int32").to_numpy()
df["prior_noshows"] = history["cum_noshows"].fillna(0).astype("int32").to_numpy()
df["has_history"] = (df["prior_appts"] > 0).astype("int8")
df["prior_noshow_rate"] = np.divide(
    df["prior_noshows"], df["prior_appts"],
    out=np.zeros(len(df), dtype=float), where=df["prior_appts"].to_numpy() > 0,
)
df[["prior_appts", "prior_noshows", "prior_noshow_rate", "has_history"]].describe().round(2)


## 4. Quick exploratory checks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["no_show"].map({0: "Attended", 1: "No-show"}).value_counts().plot.bar(
    ax=axes[0], color=["#4C9F70", "#D1495B"], rot=0
)
axes[0].set_title("Target class counts")
axes[0].set_ylabel("Appointments")
df.groupby(pd.cut(df["lead_days"], [-1, 0, 2, 7, 14, 30, 60, 1000]), observed=True)["no_show"].mean().plot(
    ax=axes[1], marker="o", color="#3B6EA8"
)
axes[1].set_title("No-show rate by booking lead time")
axes[1].set_xlabel("Days from booking to appointment")
axes[1].set_ylabel("No-show rate")
plt.tight_layout()


## 5. Features and chronological train/validation/test split

The API's `features.py` builds the same feature names and order. Lead time is capped at 180 days to match the API. We reserve the latest 15% of appointment dates for final evaluation; the earlier 15% validation period selects the follow-up threshold.


In [ ]:
TARGET = "no_show"
NUMERIC = ["Age", "lead_days", "sched_hour", "prior_appts", "prior_noshows", "prior_noshow_rate"]
BINARY = [
    "is_female", "Scholarship", "Hypertension", "Diabetes", "Alcoholism", "Handicap",
    "SMSReceived", "same_day", "has_history",
]
CATEGORICAL = ["appt_weekday", "sched_weekday", "Neighbourhood"]
FEATURES = NUMERIC + BINARY + CATEGORICAL

# Match the production feature builder's cap and avoid adding post-outcome columns.
df["lead_days"] = df["lead_days"].clip(upper=180)
ordered_dates = np.sort(df["appt_date"].unique())
train_cut = ordered_dates[int(len(ordered_dates) * 0.70)]
valid_cut = ordered_dates[int(len(ordered_dates) * 0.85)]
train = df[df["appt_date"] <= train_cut]
valid = df[(df["appt_date"] > train_cut) & (df["appt_date"] <= valid_cut)]
test = df[df["appt_date"] > valid_cut]

X_train, y_train = train[FEATURES], train[TARGET]
X_valid, y_valid = valid[FEATURES], valid[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
for split_name, split in [("Train", train), ("Validation", valid), ("Test", test)]:
    print(f"{split_name:10s}: {len(split):,} rows | {split['appt_date'].min().date()} to {split['appt_date'].max().date()} | no-show {split[TARGET].mean():.1%}")


## 6. Baseline reference

A majority-class model gives a simple reference point. Accuracy alone can look high because most patients attend, so also compare ranking and probability metrics.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
print("Baseline fitted; final baseline metrics will be reported beside Gradient Boosting.")


## 7. Train the Gradient Boosting engine

The pipeline one-hot encodes categories and passes numeric/binary features through. Histogram-based boosting bins feature values to train successive trees efficiently. Early stopping and tree limits help reduce overfitting.


In [ ]:
# Dense one-hot output is required by HistGradientBoostingClassifier.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", NUMERIC + BINARY),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        early_stopping=True,
        random_state=RANDOM_STATE,
    )),
])
model.fit(X_train, y_train)
print("Gradient Boosting model trained on the earliest time period.")


## 8. Select an action threshold on validation data

The model estimates no-show probability. Since no-shows are less common, `0.5` may miss many. Choose the threshold that reaches the recall goal on validation data; do not tune it on the final test period.


In [ ]:
p_valid = model.predict_proba(X_valid)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_valid, p_valid)
eligible = np.flatnonzero(recall[:-1] >= TARGET_RECALL)
if len(eligible) == 0:
    threshold = 0.5
    warnings.warn("Validation predictions could not reach target recall; using threshold 0.5.")
else:
    idx = eligible[-1]  # highest threshold that still reaches the target recall
    threshold = float(thresholds[idx])
    print(f"Validation threshold: {threshold:.3f} | recall {recall[idx]:.1%} | precision {precision[idx]:.1%}")


## 9. Final evaluation on the later test period

ROC-AUC measures ranking quality; average precision focuses on finding no-shows; Brier score measures probability error (lower is better). Compare both models using the same untouched test set.


In [ ]:
p_test = model.predict_proba(X_test)[:, 1]
pred_test = (p_test >= threshold).astype(int)
p_baseline = baseline.predict_proba(X_test)[:, 1]

metrics = pd.DataFrame({
    "Majority baseline": {
        "ROC-AUC": roc_auc_score(y_test, p_baseline),
        "PR-AUC": average_precision_score(y_test, p_baseline),
        "Brier": brier_score_loss(y_test, p_baseline),
    },
    "Gradient Boosting": {
        "ROC-AUC": roc_auc_score(y_test, p_test),
        "PR-AUC": average_precision_score(y_test, p_test),
        "Brier": brier_score_loss(y_test, p_test),
    },
}).T
metrics.round(3)

print(f"\nValidation-selected threshold: {threshold:.3f}")
print(f"Test no-show recall at threshold: {(pred_test[y_test.to_numpy() == 1].mean()):.1%}")
print("\nGradient Boosting classification report:")
print(classification_report(y_test, pred_test, target_names=["Attended", "No-show"], zero_division=0))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred_test), display_labels=["Attended", "No-show"]
).plot(cmap="Blues", values_format="d")
plt.title(f"Test confusion matrix (no-show threshold={threshold:.2f})")
plt.tight_layout()


## 10. Save the final model bundle for the application

The API loads `referral_success_model.joblib` and expects `model`, `features`, and a threshold on **no-show probability**. The success rate is `1 - P(no_show)`.


In [ ]:
bundle = {
    "model": model,
    "features": FEATURES,
    "threshold": float(threshold),
}
joblib.dump(bundle, MODEL_PATH)
print(f"Saved model bundle to {MODEL_PATH.resolve()}")

# Example referral shaped like the API's feature-builder output.
example = {
    "Age": 55,
    "lead_days": 21,
    "sched_hour": 9,
    "prior_appts": 2,
    "prior_noshows": 1,
    "prior_noshow_rate": 0.5,
    "is_female": 1,
    "Scholarship": 0,
    "Hypertension": 1,
    "Diabetes": 0,
    "Alcoholism": 0,
    "Handicap": 0,
    "SMSReceived": 1,
    "same_day": 0,
    "has_history": 1,
    "appt_weekday": "Monday",
    "sched_weekday": "Monday",
    "Neighbourhood": str(df["Neighbourhood"].mode().iloc[0]),
}
example_X = pd.DataFrame([example])[FEATURES]
no_show_risk = float(bundle["model"].predict_proba(example_X)[0, 1])
print(f"Example no-show risk: {no_show_risk:.1%}")
print(f"Example referral success rate: {1 - no_show_risk:.1%}")
print(f"Flag for follow-up at threshold {threshold:.3f}: {no_show_risk >= threshold}")


## 11. Interpretation and limitations

- The time split evaluates later dates from the same 2016 sample; it does not guarantee results at another clinic, year, or population.
- Threshold choice depends on follow-up capacity and the cost of missed appointments.
- Associations are not causes. SMS can act as a proxy for lead time.
- `SMS_received` must be known/planned at scoring time. If unavailable, remove it from training and the API feature contract, then retrain.
- Retrain when data, features, or scikit-learn versions change. Load joblib files only from trusted sources.
